In [2]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Configuración de rutas (asegúrate de que ROOT suba a la raíz)
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from config.paths import RAW_DIR, PROCESSED_DIR

YEARS = [2018, 2024]

# comentario 

In [3]:
in_kind_transfers = pd.read_csv(PROCESSED_DIR / "in_kind_transfers.csv",
                        dtype={'folioviv': str, 'foliohog': str, 'numren': str})


iva_hogar = pd.read_csv(PROCESSED_DIR / "iva_hogar.csv",
                        dtype={'folioviv': str, 'foliohog': str})


income_fiscal_incidence_base = pd.read_csv(PROCESSED_DIR / "income_fiscal_incidence_base.csv",
                        dtype={'folioviv': str, 'foliohog': str, 'numren':str})


demographic_household = pd.read_csv(PROCESSED_DIR / "demographic_household.csv",
                        dtype={'folioviv': str, 'foliohog': str})




/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_75403/2259054953.py:1: DtypeWarning: Columns (0: etnia) have mixed types. Specify dtype option on import or set low_memory=False.
  in_kind_transfers = pd.read_csv(PROCESSED_DIR / "in_kind_transfers.csv",


In [18]:
import pandas as pd

# ==========================================================
# 1. Número de integrantes por hogar
# ==========================================================

household_size = (
    income_fiscal_incidence_base
    .groupby(["folioviv", "foliohog", "year"])
    .size()
    .reset_index(name="household_size")
)

# ==========================================================
# 2. IVA per cápita
# ==========================================================

iva_hogar = iva_hogar.merge(
    household_size,
    on=["folioviv", "foliohog", "year"],
    how="left"
)

iva_cols = [
    "gasto_con_iva",
    "gasto_sin_iva",
    "iva_pagado",
    "gasto_total",
    "gasto_gas",
    "gasto_electricity"
]

iva_hogar[iva_cols] = iva_hogar[iva_cols].div(
    iva_hogar["household_size"],
    axis=0
)

# Si ya no la necesitas
iva_hogar = iva_hogar.drop(columns="household_size")

# ==========================================================
# 3. Base a nivel hogar
# ==========================================================

household_base = (
    demographic_household
    .merge(
        iva_hogar,
        on=["folioviv", "foliohog", "year"],
        how="left"
    )
)

# ==========================================================
# 4. Llevar variables del hogar a la base individual
# ==========================================================

individual_base = (
    income_fiscal_incidence_base
    .merge(
        household_base,
        on=["folioviv", "foliohog", "year"],
        how="left"
    )
)

# ==========================================================
# 5. Agregar transferencias en especie
# ==========================================================

# Asegurar que las llaves tengan el mismo tipo
individual_base["folioviv"] = individual_base["folioviv"].astype(str)
individual_base["foliohog"] = individual_base["foliohog"].astype(str)
individual_base["numren"] = individual_base["numren"].astype(str)

in_kind_transfers["folioviv"] = in_kind_transfers["folioviv"].astype(str)
in_kind_transfers["foliohog"] = in_kind_transfers["foliohog"].astype(str)
in_kind_transfers["numren"] = in_kind_transfers["numren"].astype(str)

# Merge
individual_base = individual_base.merge(
    in_kind_transfers,
    on=["folioviv", "foliohog", "numren", "year"],
    how="left"
)


# ==========================================================
# Resultado
# ==========================================================

individual_base.head()

individual_base.to_csv(PROCESSED_DIR / "base_final.csv", index=False)
print("Saved base_final.csv")

Saved base_final.csv


In [5]:
for name, df in {
    "income": income_fiscal_incidence_base,
    "iva": iva_hogar,
    "demo": demographic_household,
    "transfers": in_kind_transfers
}.items():
    print(name)
    print(df[["folioviv","foliohog","year"]].dtypes)
    print()

income
folioviv      str
foliohog      str
year        int64
dtype: object

iva
folioviv      str
foliohog      str
year        int64
dtype: object

demo
folioviv      str
foliohog      str
year        int64
dtype: object

transfers
folioviv      str
foliohog      str
year        int64
dtype: object



In [17]:
test = individual_base.merge(
    in_kind_transfers,
    on=["folioviv", "foliohog", "numren", "year"],
    how="left",
    indicator=True
)

test["_merge"].value_counts()

_merge
both          267486
left_only     117313
right_only         0
Name: count, dtype: int64